# 🌱 AI-Driven Gene & Trait Discovery for Plant Breeding
**PhD Starter Project — Wageningen University & Research**

This notebook is a minimal, self-contained introduction to the core ideas of the project:
1. Simulate multi-omics data (transcriptomics + metabolomics)
2. Explore and visualize the data
3. Train a simple interpretable ML model to predict a plant trait
4. Identify which genes/features drive the prediction (explainability)

All data here is **simulated** — no real dataset needed to get started.

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Install any missing packages (already available in Colab, but just in case)
!pip install shap --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

import shap

np.random.seed(42)
print('All libraries loaded successfully!')

## 🧬 Step 2 — Simulate Multi-Omics Data

We simulate:
- **Transcriptomics**: gene expression levels for 50 genes across 200 faba bean samples
- **Metabolomics**: 20 metabolite concentrations per sample
- **Trait**: a continuous phenotype (e.g. seed yield) influenced by a few key genes

In [ ]:
N_SAMPLES    = 200   # number of plant samples
N_GENES      = 50    # transcriptomics features
N_METABOLITES = 20   # metabolomics features

# --- Transcriptomics (gene expression) ---
gene_names = [f'Gene_{i+1}' for i in range(N_GENES)]
transcriptomics = pd.DataFrame(
    np.random.lognormal(mean=2.0, sigma=1.0, size=(N_SAMPLES, N_GENES)),
    columns=gene_names
)

# --- Metabolomics ---
metabolite_names = [f'Metabolite_{i+1}' for i in range(N_METABOLITES)]
metabolomics = pd.DataFrame(
    np.random.lognormal(mean=1.5, sigma=0.8, size=(N_SAMPLES, N_METABOLITES)),
    columns=metabolite_names
)

# --- Trait: seed yield (driven by 3 key genes + noise) ---
# Gene_1, Gene_5, Gene_10 are the "causal" genes we want the model to discover
trait = (
    2.5 * transcriptomics['Gene_1'] +
    1.8 * transcriptomics['Gene_5'] -
    1.2 * transcriptomics['Gene_10'] +
    0.5 * metabolomics['Metabolite_3'] +
    np.random.normal(0, 2, N_SAMPLES)  # noise
)
trait = pd.Series(trait, name='Seed_Yield')

print(f'Transcriptomics shape : {transcriptomics.shape}')
print(f'Metabolomics shape    : {metabolomics.shape}')
print(f'Trait (seed yield)    : mean={trait.mean():.2f}, std={trait.std():.2f}')

## 📊 Step 3 — Explore the Data

In [ ]:
# --- Trait distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(trait, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Trait Distribution (Seed Yield)', fontsize=13)
axes[0].set_xlabel('Seed Yield')
axes[0].set_ylabel('Count')

# --- Gene expression heatmap (first 20 genes, 30 samples) ---
sns.heatmap(
    transcriptomics.iloc[:30, :20].T,
    ax=axes[1],
    cmap='YlOrRd',
    xticklabels=False,
    yticklabels=True,
    cbar_kws={'label': 'Expression'}
)
axes[1].set_title('Gene Expression Heatmap (subset)', fontsize=13)
axes[1].set_xlabel('Samples')

plt.tight_layout()
plt.show()

In [ ]:
# --- PCA on transcriptomics ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(transcriptomics)

pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    pca_coords[:, 0], pca_coords[:, 1],
    c=trait, cmap='viridis', alpha=0.7, edgecolors='none', s=50
)
plt.colorbar(scatter, label='Seed Yield')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('PCA of Transcriptomics — colored by Trait', fontsize=13)
plt.tight_layout()
plt.show()

## 🤖 Step 4 — Train an Interpretable ML Model

We use a **Random Forest** — a strong, interpretable baseline widely used in genomics.

Input: combined transcriptomics + metabolomics features  
Output: predicted seed yield

In [ ]:
# --- Combine omics layers ---
X = pd.concat([transcriptomics, metabolomics], axis=1)
y = trait

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Train ---
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

# --- Evaluate ---
y_pred = rf.predict(X_test)
r2  = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f'R²  (higher is better): {r2:.3f}')
print(f'MSE (lower is better) : {mse:.3f}')

# --- Predicted vs Actual plot ---
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.7, color='steelblue', edgecolors='none')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=1.5)
plt.xlabel('Actual Seed Yield')
plt.ylabel('Predicted Seed Yield')
plt.title(f'Predicted vs Actual  |  R² = {r2:.3f}', fontsize=13)
plt.tight_layout()
plt.show()

## 🔍 Step 5 — Explainability: Which Genes Drive the Prediction?

We use **SHAP** (SHapley Additive exPlanations) — a gold-standard method for interpreting ML models in biology.  
This tells us *how much* each gene/metabolite contributed to each prediction.

We expect the model to rediscover **Gene_1**, **Gene_5**, **Gene_10**, and **Metabolite_3** as the top drivers.

In [ ]:
# --- SHAP values ---
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Summary bar plot — mean absolute SHAP per feature
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test, plot_type='bar', max_display=15, show=False)
plt.title('Top 15 Features by Mean |SHAP| Value', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- SHAP beeswarm plot: direction of effect ---
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.title('SHAP Beeswarm — Feature Impact Direction', fontsize=13)
plt.tight_layout()
plt.show()

## ✅ Step 6 — Verify: Did the Model Find the Right Genes?

In [ ]:
# Rank features by mean absolute SHAP value
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_test.columns
).sort_values(ascending=False)

top10 = mean_shap.head(10)

# Ground truth causal features
causal = ['Gene_1', 'Gene_5', 'Gene_10', 'Metabolite_3']

print('=== Top 10 Features by SHAP Importance ===')
for rank, (feat, val) in enumerate(top10.items(), 1):
    tag = '  ✅ CAUSAL' if feat in causal else ''
    print(f'  {rank:2d}. {feat:<20s}  SHAP={val:.3f}{tag}')

recovered = [f for f in causal if f in top10.index]
print(f'\nRecovered {len(recovered)}/{len(causal)} causal features in top 10: {recovered}')

## 📝 Summary & Next Steps

| Step | What we did |
|------|-------------|
| Data simulation | Transcriptomics (50 genes) + Metabolomics (20 metabolites), 200 samples |
| Exploration | Trait distribution, expression heatmap, PCA |
| ML model | Random Forest regressor on combined omics |
| Explainability | SHAP values to identify causal genes |

### 🚀 What to do next (PhD roadmap)

1. **Replace simulated data** with real faba bean single-cell RNA-seq / proteomics / metabolomics datasets
2. **Upgrade the model** — try gradient boosting (XGBoost/LightGBM) or graph neural networks for gene network structure
3. **Add GWAS benchmarking** — compare ML-discovered genes against GWAS hits
4. **Single-cell integration** — use `scanpy` / `AnnData` for cell-type-level analysis
5. **Multi-task learning** — predict multiple traits simultaneously

---
*Built for the WUR PhD position: AI-Driven Gene & Trait Discovery for Plant Breeding*